In [1]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
df = pd.read_csv("outputs/unified_features.csv").dropna(subset=["hr", "hrv"])

def compute_thresholds(df, n_baseline_rows=10):
    thresholds = {}
    for subj, group in df.groupby("subject"):
        baseline = group[group["ground_truth"] == "NORMAL"].head(n_baseline_rows)
        if len(baseline) < 3:
            continue
        thresholds[subj] = {
            "hr_thresh": baseline["hr"].mean() + baseline["hr"].std(),
            "hrv_thresh": max(15, baseline["hrv"].mean() - baseline["hrv"].std()),
        }
    return thresholds

thresholds = compute_thresholds(df)
print(f"Computed thresholds for {len(thresholds)} subjects")

Computed thresholds for 15 subjects


In [3]:
def classify_fixed(row):
    return "STRESSED" if (row["hrv"] < 25 and row["hr"] > 90) else "NORMAL"

def classify_personalized(row):
    subj = row["subject"]
    if subj not in thresholds:
        return classify_fixed(row)
    t = thresholds[subj]
    return "STRESSED" if (row["hrv"] < t["hrv_thresh"] and row["hr"] > t["hr_thresh"]) else "NORMAL"

df["pred_fixed"] = df.apply(classify_fixed, axis=1)
df["pred_personalized"] = df.apply(classify_personalized, axis=1)

def per_subject_acc(df, col):
    return df.groupby("subject").apply(lambda g: (g["ground_truth"] == g[col]).mean())

fixed_acc = per_subject_acc(df, "pred_fixed")
personal_acc = per_subject_acc(df, "pred_personalized")
print("Mean fixed accuracy:", fixed_acc.mean())
print("Mean personalized accuracy:", personal_acc.mean())

Mean fixed accuracy: 0.7902799433991234
Mean personalized accuracy: 0.7491539306808384


In [4]:
merged = pd.DataFrame({"fixed": fixed_acc, "personalized": personal_acc}).dropna()
stat, p = stats.wilcoxon(merged["personalized"], merged["fixed"])
print(f"Wilcoxon p={p:.4f}")
merged.to_csv("outputs/pbc_results.csv")
if p < 0.05:
    print(f"Improvement: {(merged['personalized'].mean()-merged['fixed'].mean())*100:.1f} pts, p<0.05")
else:
    print("No significant improvement — report honestly, still a valid finding")

Wilcoxon p=0.4212
No significant improvement — report honestly, still a valid finding


In [5]:
def compute_thresholds_v2(df, n_baseline_rows=25):  # increased from 10, still a defensible design choice
    thresholds = {}
    for subj, group in df.groupby("subject"):
        baseline = group[group["ground_truth"] == "NORMAL"].head(n_baseline_rows)
        if len(baseline) < 5:  # slightly stricter minimum too, since we want stable estimates
            continue
        thresholds[subj] = {
            "hr_thresh": baseline["hr"].mean() + baseline["hr"].std(),
            "hrv_thresh": max(15, baseline["hrv"].mean() - baseline["hrv"].std()),
        }
    return thresholds

thresholds_v2 = compute_thresholds_v2(df)
print(f"Computed thresholds for {len(thresholds_v2)} subjects (was {len(thresholds)} with n=10)")

def classify_personalized_v2(row):
    subj = row["subject"]
    if subj not in thresholds_v2:
        return classify_fixed(row)
    t = thresholds_v2[subj]
    return "STRESSED" if (row["hrv"] < t["hrv_thresh"] and row["hr"] > t["hr_thresh"]) else "NORMAL"

df["pred_personalized_v2"] = df.apply(classify_personalized_v2, axis=1)
personal_acc_v2 = per_subject_acc(df, "pred_personalized_v2")

merged_v2 = pd.DataFrame({"fixed": fixed_acc, "personalized": personal_acc_v2}).dropna()
stat, p = stats.wilcoxon(merged_v2["personalized"], merged_v2["fixed"])
print(f"\nMean fixed: {merged_v2['fixed'].mean():.4f}")
print(f"Mean personalized (n=25 baseline): {merged_v2['personalized'].mean():.4f}")
print(f"Wilcoxon p={p:.4f}")

Computed thresholds for 15 subjects (was 15 with n=10)

Mean fixed: 0.7903
Mean personalized (n=25 baseline): 0.7778
Wilcoxon p=0.7776
